# Used Car Price Prediction

## 1) Problem statement.

* This dataset comprises used cars sold on cardehko.com in India as well as important features of these cars.
* If user can predict the price of the car based on input features.
* Prediction results can be used to give new seller the price suggestion based on market condition.

## 2) Data Collection.
* The Dataset is collected from scrapping from cardheko webiste
* The data consists of 13 column and 15411 rows.

In [1]:
# Importing important libraries that we will be using 

import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt 
import seaborn as sns 
import plotly.express as px 
import warnings 

warnings.filterwarnings("ignore")

In [2]:
df = pd.read_csv('cardekho_imputated.csv')
df.head()

,Unnamed: 0,car_name,brand,model,vehicle_age,km_driven,seller_type,fuel_type,transmission_type,mileage,engine,max_power,seats,selling_price
0,0,Maruti Alto,Maruti,Alto,9,120000,Individual,Petrol,Manual,19.70,796,46.30,5,120000
1,1,Hyundai Grand,Hyundai,Grand,5,20000,Individual,Petrol,Manual,18.90,1197,82.00,5,550000
2,2,Hyundai i20,Hyundai,i20,11,60000,Individual,Petrol,Manual,17.00,1197,80.00,5,215000
3,3,Maruti Alto,Maruti,Alto,9,37000,Individual,Petrol,Manual,20.92,998,67.10,5,226000
4,4,Ford Ecosport,Ford,Ecosport,6,30000,Dealer,Diesel,Manual,22.77,1498,98.59,5,570000


## 3) Data cleaning
* Handling missing values
* Handling duplicate values
* Checking datatypes
* Understanding the dataset

### Handling missing values

In [3]:
df.isnull().sum()

Unnamed: 0           0
car_name             0
brand                0
model                0
vehicle_age          0
km_driven            0
seller_type          0
fuel_type            0
transmission_type    0
mileage              0
engine               0
max_power            0
seats                0
selling_price        0
dtype: int64

There are no NaN values. But there are some extra columns.
Let's deal with them

In [4]:
df.drop(columns=["Unnamed: 0", "car_name", "brand"], inplace=True, errors='ignore')

In [5]:
df.head()

,model,vehicle_age,km_driven,seller_type,fuel_type,transmission_type,mileage,engine,max_power,seats,selling_price
0,Alto,9,120000,Individual,Petrol,Manual,19.70,796,46.30,5,120000
1,Grand,5,20000,Individual,Petrol,Manual,18.90,1197,82.00,5,550000
2,i20,11,60000,Individual,Petrol,Manual,17.00,1197,80.00,5,215000
3,Alto,9,37000,Individual,Petrol,Manual,20.92,998,67.10,5,226000
4,Ecosport,6,30000,Dealer,Diesel,Manual,22.77,1498,98.59,5,570000


In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15411 entries, 0 to 15410
Data columns (total 11 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   model              15411 non-null  object 
 1   vehicle_age        15411 non-null  int64  
 2   km_driven          15411 non-null  int64  
 3   seller_type        15411 non-null  object 
 4   fuel_type          15411 non-null  object 
 5   transmission_type  15411 non-null  object 
 6   mileage            15411 non-null  float64
 7   engine             15411 non-null  int64  
 8   max_power          15411 non-null  float64
 9   seats              15411 non-null  int64  
 10  selling_price      15411 non-null  int64  
dtypes: float64(2), int64(5), object(4)
memory usage: 1.3+ MB


We will now separate features based on the type of data they store 

In [7]:
numerical_features = [features for features in df.columns if df[features].dtype != 'O']
categorical_features = [features for features in df.columns if df[features].dtype == 'O']

# Numerical features can be further differentiated into discrete and continuous features 
discrete_features = [features for features in numerical_features if len(df[features].unique()) <= 25]
continuous_features = [features for features in numerical_features if len(df[features].unique()) > 25]

In [8]:
print(f"Number of numerical features: {len(numerical_features)}")
print(f"Number of categorical features: {len(categorical_features)}")
print(f"Number of discrete features: {len(discrete_features)}")
print(f"Number of continuous features: {len(continuous_features)}")

Number of numerical features: 7
Number of categorical features: 4
Number of discrete features: 2
Number of continuous features: 5


In [9]:
# Independent and dependent features 
X = df.drop(columns=['selling_price'])
y = df['selling_price']

In [10]:
X.head()

,model,vehicle_age,km_driven,seller_type,fuel_type,transmission_type,mileage,engine,max_power,seats
0,Alto,9,120000,Individual,Petrol,Manual,19.70,796,46.30,5
1,Grand,5,20000,Individual,Petrol,Manual,18.90,1197,82.00,5
2,i20,11,60000,Individual,Petrol,Manual,17.00,1197,80.00,5
3,Alto,9,37000,Individual,Petrol,Manual,20.92,998,67.10,5
4,Ecosport,6,30000,Dealer,Diesel,Manual,22.77,1498,98.59,5


## Feature Encoding and Scaling
**One Hot Encoding for Columns which had lesser unique values and not ordinal**
* One hot encoding is a process by which categorical variables are converted into a form that could be provided to ML algorithms to do a better job in prediction.

In [11]:
df['model'].value_counts()

model
i20            906
Swift Dzire    890
Swift          781
Alto           778
City           757
              ... 
Ghibli           1
Altroz           1
GTC4Lusso        1
Aura             1
Gurkha           1
Name: count, Length: 120, dtype: int64

In [12]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
X['model'] = le.fit_transform(X['model'])
X.head()

,model,vehicle_age,km_driven,seller_type,fuel_type,transmission_type,mileage,engine,max_power,seats
0,7,9,120000,Individual,Petrol,Manual,19.70,796,46.30,5
1,54,5,20000,Individual,Petrol,Manual,18.90,1197,82.00,5
2,118,11,60000,Individual,Petrol,Manual,17.00,1197,80.00,5
3,7,9,37000,Individual,Petrol,Manual,20.92,998,67.10,5
4,38,6,30000,Dealer,Diesel,Manual,22.77,1498,98.59,5


In [13]:
X['engine'].value_counts()

engine
1197    2436
1248    1668
998     1174
1498    1095
2179     669
        ... 
3598       1
3628       1
1330       1
3855       1
4163       1
Name: count, Length: 110, dtype: int64

In [14]:
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer

num_columns=X.select_dtypes(exclude='object').columns
onehot_columns=['seller_type', 'fuel_type', 'transmission_type']

numeric_transformer = StandardScaler()
onh_transformer = OneHotEncoder(drop='first')

preprocessor = ColumnTransformer(
    [
        ("OneHotEncoder", onh_transformer, onehot_columns),
        ("StandardScaler", numeric_transformer, num_columns)
    ], remainder='passthrough'
)

In [15]:
X = preprocessor.fit_transform(X)
pd.DataFrame(X)

,0,1,2,3,4,5,6,7,8,9,10,11,12,13
0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,-1.519714,0.983562,1.247335,-0.000276,-1.324259,-1.263352,-0.403022
1,1.0,0.0,0.0,0.0,0.0,1.0,1.0,-0.225693,-0.343933,-0.690016,-0.192071,-0.554718,-0.432571,-0.403022
2,1.0,0.0,0.0,0.0,0.0,1.0,1.0,1.536377,1.647309,0.084924,-0.647583,-0.554718,-0.479113,-0.403022
3,1.0,0.0,0.0,0.0,0.0,1.0,1.0,-1.519714,0.983562,-0.360667,0.292211,-0.936610,-0.779312,-0.403022
4,0.0,0.0,1.0,0.0,0.0,0.0,1.0,-0.666211,-0.012060,-0.496281,0.735736,0.022918,-0.046502,-0.403022
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15406,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.508844,0.983562,-0.869744,0.026096,-0.767733,-0.757204,-0.403022
15407,0.0,0.0,0.0,0.0,0.0,1.0,1.0,-0.556082,-1.339555,-0.728763,-0.527711,-0.216964,-0.220803,2.073444
15408,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.407551,-0.012060,0.220539,0.344954,0.022918,0.068225,-0.403022
15409,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.426247,-0.343933,72.541850,-0.887326,1.329794,0.917158,2.073444


In [16]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train.shape, X_test.shape

((12328, 14), (3083, 14))

In [17]:
X_train

array([[ 0.        ,  0.        ,  1.        , ...,  1.75390551,
         2.66249771, -0.40302241],
       [ 1.        ,  0.        ,  0.        , ..., -0.55087963,
        -0.38602844, -0.40302241],
       [ 0.        ,  0.        ,  1.        , ...,  0.89033072,
         3.27453006, -0.40302241],
       ...,
       [ 1.        ,  0.        ,  0.        , ..., -0.9366097 ,
        -0.78070786, -0.40302241],
       [ 0.        ,  0.        ,  0.        , ..., -0.55471774,
        -0.43582879, -0.40302241],
       [ 1.        ,  0.        ,  0.        , ..., -0.04616815,
         0.06194201, -0.40302241]])

## Model training and selection 

In [18]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression, Ridge,Lasso
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

In [19]:
def evaluate_model(true, predicted):
    mae = mean_absolute_error(true, predicted)
    mse = mean_squared_error(true, predicted)
    rmse = np.sqrt(mse)
    r2_square = r2_score(true, predicted)
    return mae, rmse, r2_square

In [20]:
models = {
    "Linear Regression" : LinearRegression(),
    "Lasso" : Lasso(), 
    "Ridge" : Ridge(), 
    "K-Neighbours Regressor" : KNeighborsRegressor(), 
    "Decision Tree Regressor" : DecisionTreeRegressor(), 
    "Random Forest Regressor" : RandomForestRegressor() 
}

for i in range(len(list(models))): 
    model = list(models.values())[i]
    model.fit(X_train, y_train) # Train the model

    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    model_train_mae, model_train_rmse, model_train_r2  = evaluate_model(y_train, y_train_pred)
    model_test_mae, model_test_rmse, model_test_r2 = evaluate_model(y_test, y_test_pred) 

    print(list(models.keys())[i])

    print('Model performance for Training set')
    print("- Root Mean Squared Error: {:.4f}".format(model_train_rmse))
    print("- Mean Absolute Error: {:.4f}".format(model_train_mae))
    print("- R2 Score: {:.4f}".format(model_train_r2))

    print('----------------------------------')
    
    print('Model performance for Test set')
    print("- Root Mean Squared Error: {:.4f}".format(model_test_rmse))
    print("- Mean Absolute Error: {:.4f}".format(model_test_mae))
    print("- R2 Score: {:.4f}".format(model_test_r2))
    
    print('='*35)
    print('\n')

Linear Regression
Model performance for Training set
- Root Mean Squared Error: 553855.6665
- Mean Absolute Error: 268101.6071
- R2 Score: 0.6218
----------------------------------
Model performance for Test set
- Root Mean Squared Error: 502543.5930
- Mean Absolute Error: 279618.5794
- R2 Score: 0.6645


Lasso
Model performance for Training set
- Root Mean Squared Error: 553855.6710
- Mean Absolute Error: 268099.2226
- R2 Score: 0.6218
----------------------------------
Model performance for Test set
- Root Mean Squared Error: 502542.6696
- Mean Absolute Error: 279614.7461
- R2 Score: 0.6645


Ridge
Model performance for Training set
- Root Mean Squared Error: 553856.3160
- Mean Absolute Error: 268059.8015
- R2 Score: 0.6218
----------------------------------
Model performance for Test set
- Root Mean Squared Error: 502533.8230
- Mean Absolute Error: 279557.2169
- R2 Score: 0.6645


K-Neighbours Regressor
Model performance for Training set
- Root Mean Squared Error: 325881.5005
- Mean

In [21]:
#Initialize few parameter for Hyperparamter tuning

knn_params = {"n_neighbors": [2, 3, 10, 20, 40, 50]}
rf_params = {"max_depth": [5, 8, 15, None, 10],
             "max_features": [5, 7, "auto", 8],
             "min_samples_split": [2, 8, 15, 20],
             "n_estimators": [100, 200, 500, 1000]}

In [22]:
# Models list for Hyperparameter tuning 

randomcv_models = [
    ('KNN', KNeighborsRegressor(), knn_params), 
    ('RF', RandomForestRegressor(), rf_params)
]

In [23]:
# Hyperparameter training 

from sklearn.model_selection import RandomizedSearchCV

model_param = {}
for name, model, params in randomcv_models: 
    random = RandomizedSearchCV(
        estimator=model, cv=3, n_iter=100, param_distributions=params, n_jobs= -1, verbose=2
    )
    random.fit(X_train, y_train)
    model_param[name] = random.best_params_

for model_name in model_param:
    print(f"---------------- Best Params for {model_name} -------------------")
    print(model_param[model_name])

Fitting 3 folds for each of 6 candidates, totalling 18 fits
Fitting 3 folds for each of 100 candidates, totalling 300 fits
[CV] END ......................................n_neighbors=2; total time=   0.1s
[CV] END .....................................n_neighbors=20; total time=   0.2s
[CV] END max_depth=15, max_features=8, min_samples_split=2, n_estimators=1000; total time=  11.7s


Exception ignored in: <function ResourceTracker.__del__ at 0x102f35bc0>
Traceback (most recent call last):
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 116, in _stop_locked
ChildProcessError: [Errno 10] No child processes


[CV] END ......................................n_neighbors=2; total time=   0.0s
[CV] END .....................................n_neighbors=10; total time=   0.1s
[CV] END .....................................n_neighbors=20; total time=   0.1s
[CV] END .....................................n_neighbors=50; total time=   0.2s
[CV] END max_depth=15, max_features=8, min_samples_split=2, n_estimators=1000; total time=  11.7s


Exception ignored in: <function ResourceTracker.__del__ at 0x102a7dbc0>
Traceback (most recent call last):
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 116, in _stop_locked
ChildProcessError: [Errno 10] No child processes


[CV] END ......................................n_neighbors=3; total time=   0.1s
[CV] END .....................................n_neighbors=50; total time=   0.2s
[CV] END max_depth=15, max_features=8, min_samples_split=2, n_estimators=1000; total time=  11.9s


...

[CV] END max_depth=None, max_features=5, min_samples_split=2, n_estimators=200; total time=   2.2s
[CV] END max_depth=8, max_features=5, min_samples_split=8, n_estimators=500; total time=   2.6s
[CV] END max_depth=15, max_features=8, min_samples_split=8, n_estimators=1000; total time=  10.9s
[CV] END max_depth=8, max_features=8, min_samples_split=15, n_estimators=100; total time=   0.8s
[CV] END max_depth=5, max_features=5, min_samples_split=20, n_estimators=200; total time=   0.8s
[CV] END max_depth=5, max_features=5, min_samples_split=20, n_estimators=200; total time=   0.8s
[CV] END max_depth=8, max_features=7, min_samples_split=2, n_estimators=500; total time=   3.3s
[CV] END max_depth=10, max_features=auto, min_samples_split=2, n_estimators=200; total time=   0.0s
[CV] END max_depth=10, max_features=auto, min_samples_split=2, n_estimators=200; total time=   0.0s
[CV] END max_depth=10, max_features=auto, min_samples_split=2, n_estimators=200; total time=   0.0s
[CV] END max_depth=8

Exception ignored in: <function ResourceTracker.__del__ at 0x104b69bc0>
Traceback (most recent call last):
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 116, in _stop_locked
ChildProcessError: [Errno 10] No child processes


[CV] END ......................................n_neighbors=2; total time=   0.1s
[CV] END .....................................n_neighbors=40; total time=   0.2s
[CV] END max_depth=None, max_features=7, min_samples_split=15, n_estimators=1000; total time=   9.1s
[CV] END max_depth=15, max_features=8, min_samples_split=2, n_estimators=100; total time=   1.1s
[CV] END max_depth=15, max_features=8, min_samples_split=2, n_estimators=100; total time=   1.2s
[CV] END max_depth=8, max_features=5, min_samples_split=15, n_estimators=1000; total time=   5.1s
[CV] END max_depth=None, max_features=8, min_samples_split=8, n_estimators=200; total time=   2.3s
[CV] END max_depth=8, max_features=8, min_samples_split=8, n_estimators=1000; total time=   7.3s
[CV] END max_depth=None, max_features=8, min_samples_split=15, n_estimators=200; total time=   2.1s
[CV] END max_depth=15, max_features=5, min_samples_split=8, n_estimators=500; total time=   3.7s
[CV] END max_depth=8, max_features=5, min_samples_sp

Exception ignored in: <function ResourceTracker.__del__ at 0x10303dbc0>
Traceback (most recent call last):
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 116, in _stop_locked
ChildProcessError: [Errno 10] No child processes


---------------- Best Params for KNN -------------------
{'n_neighbors': 10}
---------------- Best Params for RF -------------------
{'n_estimators': 500, 'min_samples_split': 2, 'max_features': 5, 'max_depth': 15}


In [24]:
# Retraining the models with best parameters 

models = {
    "K-Neighbours Regressor": KNeighborsRegressor(n_neighbors=10, n_jobs=-1),
    "Random Forest Regressor": RandomForestRegressor(n_estimators=500, min_samples_split=2, max_features=7, max_depth=15, n_jobs=-1)
}
trained_models = {}
for name, model in models.items(): 

    model.fit(X_train, y_train) # Model training

    trained_models[name] = model

    # Predictions
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    # Metrics for model performance 
    model_train_mae, model_train_rmse, model_train_r2 = evaluate_model(y_train, y_train_pred)
    model_test_mae, model_test_rmse, model_test_r2 = evaluate_model(y_test, y_test_pred)

    print(name)

    print('Model performance for Training set')
    print("- Root Mean Squared Error: {:.4f}".format(model_train_rmse))
    print("- Mean Absolute Error: {:.4f}".format(model_train_mae))
    print("- R2 Score: {:.4f}".format(model_train_r2))

    print('----------------------------------')
    
    print('Model performance for Test set')
    print("- Root Mean Squared Error: {:.4f}".format(model_test_rmse))
    print("- Mean Absolute Error: {:.4f}".format(model_test_mae))
    print("- R2 Score: {:.4f}".format(model_test_r2))
    
    print('='*35)
    print('\n')

K-Neighbours Regressor
Model performance for Training set
- Root Mean Squared Error: 363444.3871
- Mean Absolute Error: 103455.4916
- R2 Score: 0.8371
----------------------------------
Model performance for Test set
- Root Mean Squared Error: 263863.6616
- Mean Absolute Error: 117441.9802
- R2 Score: 0.9075


Random Forest Regressor
Model performance for Training set
- Root Mean Squared Error: 137674.8924
- Mean Absolute Error: 54326.3974
- R2 Score: 0.9766
----------------------------------
Model performance for Test set
- Root Mean Squared Error: 213064.2554
- Mean Absolute Error: 97793.0767
- R2 Score: 0.9397




## Deployment and real world exposure 
* In this section of the notebook we will make pickle files that will help us deploy this model

In [26]:
import pickle
best_model = trained_models['Random Forest Regressor']
with open('random_forest_regressor.pkl', 'wb') as f:
    pickle.dump(best_model, f) # Pickle file for trained model

In [27]:
# pickle files for preprocessor and label encoder 

with open('preprocessor.pkl', 'wb') as f:
    pickle.dump(preprocessor, f) 

with open('model_label_encoder.pkl', 'wb') as f:
    pickle.dump(le, f) 

In [33]:
car_models = le.classes_

In [34]:
import json 

with open("car_models.json", "w") as f:
    json.dump(car_models.tolist(), f)

In [35]:
import streamlit
import pandas
import numpy
import sklearn

print("streamlit:", streamlit.__version__)
print("pandas:", pandas.__version__)
print("numpy:", numpy.__version__)
print("sklearn:", sklearn.__version__)

streamlit: 1.45.1
pandas: 2.2.3
numpy: 2.1.3
sklearn: 1.6.1


In [36]:
import os
print(os.path.getsize('random_forest_regressor.pkl') / (1024*1024), "MB")
print(os.path.getsize('preprocessor.pkl') / (1024*1024), "MB")
print(os.path.getsize('model_label_encoder.pkl') / (1024*1024), "MB")

229.79032135009766 MB
0.0022172927856445312 MB
0.0011625289916992188 MB
